# Phase 4 : Validation avec Great Expectations

Suite d'expectations automatisées couvrant les 6 piliers :
- **Complétude** : colonnes non nulles, taux minimum
- **Exactitude** : types de données
- **Validité** : plages autorisées  
- **Cohérence** : comparaisons inter-colonnes
- **Unicité** : clés primaires
- **Actualité** : dates récentes, pas de futures

In [1]:
import great_expectations as gx
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('../.env')  

DB_HOST = os.getenv('DB_HOST', 'postgres')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'games_db')
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(CONNECTION_STRING)
    df = pd.read_sql("SELECT * FROM games_gold", engine)
    print(f"Données chargées depuis PostgreSQL: {len(df):,} lignes × {len(df.columns)} colonnes")
except Exception as e:
    print(f"Erreur de connexion PostgreSQL: {e}")
    # Fallback vers fichier CSV si base indisponible
    df = pd.read_csv('../data/Gold/games_gold.csv')
    print(f"Utilisation du fichier CSV en fallback: {len(df):,} lignes")

# Créer un contexte pour gx
context = gx.get_context(mode="file", project_root_dir="..")

Données chargées depuis PostgreSQL: 122,610 lignes × 39 colonnes


In [2]:
# Configuration de great expectations

datasource_name = "games_datasource"
asset_name = "games_gold"
suite_name = "games_quality_suite"

# Datasource
try:
    datasource = context.data_sources.get(datasource_name)
except:
    datasource = context.data_sources.add_pandas(datasource_name)

# Asset
try:
    data_asset = datasource.get_asset(asset_name)
except:
    data_asset = datasource.add_dataframe_asset(name=asset_name)

# Batch definition
try:
    batch_definition = data_asset.get_batch_definition("games_batch")
except:
    batch_definition = data_asset.add_batch_definition_whole_dataframe("games_batch")

# liste des règles à vérifier (vide pour l'instant)
try:
    suite = context.suites.get(suite_name)
    context.suites.delete(suite_name)
    suite = gx.ExpectationSuite(name=suite_name)
    suite = context.suites.add(suite)
except:
    suite = gx.ExpectationSuite(name=suite_name)
    suite = context.suites.add(suite)

batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
# validator : c'est l'outil qui exécute les tests sur les données
validator = context.get_validator(batch=batch, expectation_suite=suite)

## 1. Complétude 

In [3]:
# AppID ne doit jamais être null (clé primaire)
validator.expect_column_values_to_not_be_null("app_id")

# Name ne doit jamais être null
validator.expect_column_values_to_not_be_null("name")

# Release date doit avoir au moins 95% de valeurs non nulles
validator.expect_column_values_to_not_be_null("release_date", mostly=0.95)

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "release_date",
      "mostly": 0.95
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 2. Unicité

In [4]:
# AppID doit être unique
validator.expect_column_values_to_be_unique("app_id")

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_unique",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "app_id"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 3. Exactitude

In [5]:
# Price doit être de type float
validator.expect_column_values_to_be_of_type("price", "float64")

# AppID doit être de type entier
validator.expect_column_values_to_be_of_type("app_id", "int64")

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_of_type",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "app_id",
      "type_": "int64"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 4. Validité

In [6]:
# Price doit être >= 0
validator.expect_column_values_to_be_between("price", min_value=0, max_value=1000)

# Required age entre 0 et 21
validator.expect_column_values_to_be_between("required_age", min_value=0, max_value=21)

# Metacritic score entre 0 et 100
validator.expect_column_values_to_be_between("metacritic_score", min_value=0, max_value=100)

# Review score entre 0 et 100
validator.expect_column_values_to_be_between("review_score", min_value=0, max_value=100)

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "review_score",
      "min_value": 0.0,
      "max_value": 100.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 5. Cohérence 

In [7]:
# Owners_max >= Owners_min
validator.expect_column_pair_values_A_to_be_greater_than_B(
    "owners_max",
    "owners_min",
    or_equal=True
)

# Positive reviews <= Total reviews
validator.expect_column_pair_values_A_to_be_greater_than_B(
    "total_reviews",
    "positive",
    or_equal=True
)

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_pair_values_a_to_be_greater_than_b",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column_A": "total_reviews",
      "column_B": "positive",
      "or_equal": true
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 6. Actualité

In [8]:
# Release year entre 1980 et 2026 (pas de dates futures)
validator.expect_column_values_to_be_between("release_year", min_value=1980, max_value=2026)

/opt/conda/lib/python3.11/site-packages/great_expectations/expectations/expectation.py:1633: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "release_year",
      "min_value": 1980.0,
      "max_value": 2026.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## Exécution de la Validation

In [9]:
# Sauvegarde de la suite
suite = validator.get_expectation_suite()
suite.save()

# Création du ValidationDefinition et Checkpoint pour l'exécution automatique
validation_name = "games_validation"
try:
    validation_def = context.validation_definitions.get(validation_name)
except:
    validation_def = gx.ValidationDefinition(
        name=validation_name,
        data=batch_definition,
        suite=suite,
    )
    validation_def = context.validation_definitions.add(validation_def)

checkpoint_name = "games_checkpoint"
try:
    checkpoint = context.checkpoints.get(checkpoint_name)
except:
    checkpoint = gx.Checkpoint(
        name=checkpoint_name,
        validation_definitions=[validation_def],
        # actions sur échec : stockage des résultats + mise à jour Data Docs
        # on pourrait aussi ajouter SlackNotificationAction pour les alertes
    )
    checkpoint = context.checkpoints.add(checkpoint)

# Exécution
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})

# Mise à jour des Data Docs (rapport HTML interactif)
context.build_data_docs()
context.open_data_docs()

print(f"Validation exécutée : {len(suite.expectations)} expectations")

Calculating Metrics:   0%|          | 0/71 [00:00<?, ?it/s]

Validation exécutée : 13 expectations


In [10]:
# résultats du checkpoint
results = list(checkpoint_result.run_results.values())[0]
stats = results.statistics
result_list = results.results


print(f"Expectations évaluées: {stats['evaluated_expectations']}")
print(f"Succès: {stats['successful_expectations']}")
print(f"Échecs: {stats['unsuccessful_expectations']}")
print(f"Taux de succès: {stats['success_percent']:.1f}%")

print("\nDétails :")
for result in result_list:
    status = "OK" if result.success else "FAILED"
    exp_type = result.expectation_config.type.replace("expect_column_", "").replace("expect_", "")
    kwargs = result.expectation_config.kwargs
    column = kwargs.get('column', kwargs.get('column_A', 'N/A'))
    print(f"   {status} {column}: {exp_type}")

Expectations évaluées: 13
Succès: 13
Échecs: 0
Taux de succès: 100.0%

Détails :
   OK app_id: values_to_not_be_null
   OK app_id: values_to_be_unique
   OK app_id: values_to_be_of_type
   OK name: values_to_not_be_null
   OK release_date: values_to_not_be_null
   OK price: values_to_be_of_type
   OK price: values_to_be_between
   OK required_age: values_to_be_between
   OK metacritic_score: values_to_be_between
   OK review_score: values_to_be_between
   OK owners_max: pair_values_a_to_be_greater_than_b
   OK total_reviews: pair_values_a_to_be_greater_than_b
   OK release_year: values_to_be_between


In [11]:
# DataFrame récapitulatif pour visualisation
import pandas as pd

rapport = []
for r in result_list:
    exp = r.expectation_config
    exp_type = exp.type.replace("expect_column_", "").replace("expect_", "")
    kwargs = exp.kwargs
    column = kwargs.get('column', kwargs.get('column_A', '-'))

    if 'not_be_null' in exp.type:
        pilier = "Complétude"
    elif 'unique' in exp.type:
        pilier = "Unicité"
    elif 'be_of_type' in exp.type:
        pilier = "Exactitude"
    elif 'be_between' in exp.type:
        if 'year' in column.lower():
            pilier = "Actualité"
        else:
            pilier = "Validité"
    elif 'greater_than' in exp.type or 'pair' in exp.type:
        pilier = "Cohérence"
    else:
        pilier = "Autre"

    rapport.append({
        "Pilier": pilier,
        "Colonne": column,
        "Test": exp_type,
        "Résultat": "Succès" if r.success else "Échec"
    })

df_rapport = pd.DataFrame(rapport)
df_rapport

,Pilier,Colonne,Test,Résultat
0,Complétude,app_id,values_to_not_be_null,Succès
1,Unicité,app_id,values_to_be_unique,Succès
2,Exactitude,app_id,values_to_be_of_type,Succès
3,Complétude,name,values_to_not_be_null,Succès
4,Complétude,release_date,values_to_not_be_null,Succès
5,Exactitude,price,values_to_be_of_type,Succès
6,Validité,price,values_to_be_between,Succès
7,Validité,required_age,values_to_be_between,Succès
8,Validité,metacritic_score,values_to_be_between,Succès
9,Validité,review_score,values_to_be_between,Succès


## Validation des tables relationnelles Gold

In [12]:
# Charger les tables relationnelles depuis PostgreSQL
try:
    df_tags = pd.read_sql("SELECT * FROM game_tags", engine)
    df_genres = pd.read_sql("SELECT * FROM game_genres", engine) 
    df_developers = pd.read_sql("SELECT * FROM game_developers", engine)
    df_publishers = pd.read_sql("SELECT * FROM game_publishers", engine)
    print(f"Tags: {len(df_tags):,} lignes")
    print(f"Genres: {len(df_genres):,} lignes") 
    print(f"Developers: {len(df_developers):,} lignes")
    print(f"Publishers: {len(df_publishers):,} lignes")
except Exception as e:
    print(f"Erreur PostgreSQL pour tables relationnelles: {e}")
    # Fallback vers fichiers CSV
    df_tags = pd.read_csv('../data/Gold/game_tags.csv')
    df_genres = pd.read_csv('../data/Gold/game_genres.csv')
    df_developers = pd.read_csv('../data/Gold/game_developers.csv')
    df_publishers = pd.read_csv('../data/Gold/game_publishers.csv')
    print(f"Utilisation des fichiers CSV en fallback")

Tags: 1,219,291 lignes
Genres: 329,318 lignes
Developers: 127,802 lignes
Publishers: 121,071 lignes


In [13]:
def validate_relational_table(df_table, table_name, id_col, value_col, main_df):
    """Valide une table relationnelle Gold"""
    results = {"table": table_name, "tests": [], "passed": 0, "failed": 0}

    # Test 1: Pas de null dans la colonne ID
    null_ids = df_table[id_col].isnull().sum()
    passed = null_ids == 0
    results["tests"].append(f"{id_col} not null: {'OK' if passed else 'FAILED'}")
    results["passed" if passed else "failed"] += 1

    # Test 2: Pas de null dans la colonne valeur
    null_vals = df_table[value_col].isnull().sum()
    passed = null_vals == 0
    results["tests"].append(f"{value_col} not null: {'OK' if passed else 'FAIL'}")
    results["passed" if passed else "failed"] += 1

    # Test 3: Pas de valeurs vides
    empty_vals = (df_table[value_col] == '').sum()
    passed = empty_vals == 0
    results["tests"].append(f"{value_col} not empty: {'OK' if passed else 'FAILED'}")
    results["passed" if passed else "failed"] += 1

    # Test 4: tous les AppID existent dans games_gold
    orphan_ids = set(df_table[id_col]) - set(main_df['app_id'])
    passed = len(orphan_ids) == 0
    results["tests"].append(f"Référential integrity: {'OK' if passed else 'FAILED'} ({len(orphan_ids)} orphans)")
    results["passed" if passed else "failed"] += 1

    return results


tables_results = []
tables_results.append(validate_relational_table(df_tags, "game_tags", "app_id", "tag", df))
tables_results.append(validate_relational_table(df_genres, "game_genres", "app_id", "genre", df))
tables_results.append(validate_relational_table(df_developers, "game_developers", "app_id", "developer", df))
tables_results.append(validate_relational_table(df_publishers, "game_publishers", "app_id", "publisher", df))

for r in tables_results:
    print(f"\n{r['table']}")
    for t in r['tests']:
        print(f"   {t}")
    print(f"   -> {r['passed']}/{r['passed']+r['failed']} tests passés")


game_tags
   app_id not null: OK
   tag not null: OK
   tag not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_genres
   app_id not null: OK
   genre not null: OK
   genre not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_developers
   app_id not null: OK
   developer not null: OK
   developer not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_publishers
   app_id not null: OK
   publisher not null: OK
   publisher not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés


In [14]:
# Résumé global
total_passed = stats['successful_expectations'] + sum(r['passed'] for r in tables_results)
total_tests = stats['evaluated_expectations'] + sum(r['passed']+r['failed'] for r in tables_results)

print(f"\nTable principale (games_gold):")
print(f"   {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations ({stats['success_percent']:.0f}%)")

print(f"\nTables relationnelles:")
for r in tables_results:
    pct = r['passed']/(r['passed']+r['failed'])*100
    print(f"   {r['table']}: {r['passed']}/{r['passed']+r['failed']} ({pct:.0f}%)")


print(f"\nTOTAL: {total_passed}/{total_tests} tests passés ({total_passed/total_tests*100:.0f}%)")



Table principale (games_gold):
   13/13 expectations (100%)

Tables relationnelles:
   game_tags: 4/4 (100%)
   game_genres: 4/4 (100%)
   game_developers: 4/4 (100%)
   game_publishers: 4/4 (100%)

TOTAL: 29/29 tests passés (100%)


In [15]:
# Résumé par pilier
summary = df_rapport.groupby('Pilier').agg(
    Tests=('Résultat', 'count'),
    Succès=('Résultat', lambda x: (x == 'Succès').sum())
)
summary['Taux'] = (summary['Succès'] / summary['Tests'] * 100).round(1).astype(str) + '%'
print(summary.to_string())
print(f"\nTOTAL: {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations réussies ({stats['success_percent']:.0f}%)")

            Tests  Succès    Taux
Pilier                           
Actualité       1       1  100.0%
Cohérence       2       2  100.0%
Complétude      3       3  100.0%
Exactitude      2       2  100.0%
Unicité         1       1  100.0%
Validité        4       4  100.0%

TOTAL: 13/13 expectations réussies (100%)


## Récapitulatif des 13 Expectations

| Pilier | Expectation | Colonne |
|--------|-------------|---------|
| Complétude | not_be_null 100% | AppID |
| Complétude | not_be_null 100% | Name |
| Complétude | not_be_null 95% | Release date |
| Unicité | be_unique | AppID |
| Exactitude | be_of_type float64 | Price |
| Exactitude | be_of_type int64 | AppID |
| Validité | be_between 0-1000 | Price |
| Validité | be_between 0-21 | Required age |
| Validité | be_between 0-100 | Metacritic score |
| Validité | be_between 0-100 | Review score |
| Cohérence | A >= B | Owners_max vs Owners_min |
| Cohérence | A >= B | Total reviews vs Positive |
| Actualité | be_between 1980-2026 | Release year |